# 創薬仮説生成システム — スタンドアロン版

すべての処理（データ収集・PPI・エンリッチメント・仮説生成・レポート）を1つのノートブックに収録。

**前提条件:**
```bash
pip install requests networkx matplotlib ipywidgets gprofiler-official
ollama serve          # 別ターミナルで起動
ollama pull qwen2.5:14b
```

**実行順序:** Step 1 → Step 2 → Step 3 → Step 4

In [1]:
# ============================================================
# ライブラリインポート（初回のみ実行）
# ============================================================
import json, re, time, math, os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("networkx 未インストール: pip install networkx")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib 未インストール: pip install matplotlib")

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

print("✓ ライブラリ読み込み完了")

✓ ライブラリ読み込み完了


In [2]:
# ============================================================
# Step 1: 設定
# ============================================================
MODEL = "qwen2.5:14b"   # Ollama モデル名（ollama list で確認）
LANG  = "en"            # "en" or "ja"
REPORTS_DIR = Path("reports")

CONTEXT_CONFIG = dict(
    max_papers=5, abstract_chars=600,
    max_drugs=8, max_gwas=5, max_clinvar=5,
    max_interactions=10, max_trials=6, max_reactome=10,
    gtex_top_n=5, hpa_top_n=8, max_dgidb=8,
    uniprot_chars=500, uniprot_keywords=10, uniprot_go_terms=8,
)

# ローカルキャッシュ設定（PPI）
PPI_CACHE_DIR = Path("ppi_cache")
PPI_CACHE_TTL = 3 * 24 * 3600   # 3日間（IntAct/Reactome）
SIGNOR_CACHE_TTL = 7 * 24 * 3600  # 7日間（SIGNOR全TSV）

print(f"モデル: {MODEL}  言語: {LANG}")

モデル: qwen2.5:14b  言語: en


In [3]:
# ============================================================
# LLM クライアント（Ollama）
# ============================================================
class OllamaClient:
    def __init__(self, model=MODEL, base_url="http://localhost:11434"):
        self.model = model
        self.base_url = base_url

    def generate(self, prompt, temperature=0.3, max_tokens=4096,
                 stream_callback=None, format=None):
        use_stream = stream_callback is not None
        payload = {
            "model": self.model, "prompt": prompt,
            "stream": use_stream,
            "options": {"temperature": temperature, "num_predict": max_tokens},
        }
        if format:
            payload["format"] = format
        r = requests.post(f"{self.base_url}/api/generate",
                          json=payload, stream=use_stream, timeout=300)
        r.raise_for_status()
        if use_stream:
            full = []
            for line in r.iter_lines():
                if not line: continue
                chunk = json.loads(line)
                tok = chunk.get("response", "")
                if tok:
                    full.append(tok)
                    stream_callback(tok)
                if chunk.get("done"): break
            return "".join(full)
        return r.json().get("response", "")

    def is_available(self):
        try:
            return requests.get(f"{self.base_url}/api/tags", timeout=3).status_code == 200
        except: return False

llm = OllamaClient(model=MODEL)
if llm.is_available():
    print(f"✓ Ollama 接続OK  モデル: {MODEL}")
else:
    print("✗ Ollama に接続できません。`ollama serve` を起動してください")

✓ Ollama 接続OK  モデル: qwen2.5:14b


In [4]:
# ============================================================
# PubMed — 遺伝子・疾患シノニム取得 + 文献検索
# ============================================================
_EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def get_gene_synonyms(gene_symbol):
    """NCBI Gene + UniProt から遺伝子シノニムとタンパク質名を取得。"""
    synonyms = [gene_symbol]
    seen = {gene_symbol.upper()}
    def add(name):
        name = str(name).strip()
        if name and name.upper() not in seen and len(name) >= 2:
            seen.add(name.upper()); synonyms.append(name)
    # NCBI Gene
    try:
        r = requests.get(f"{_EUTILS}/esearch.fcgi", params={
            "db": "gene", "term": f"{gene_symbol}[Gene Name] AND Homo sapiens[Organism]",
            "retmax": 1, "retmode": "json"}, timeout=10)
        ids = r.json().get("esearchresult", {}).get("idlist", [])
        if ids:
            time.sleep(0.3)
            r2 = requests.get(f"{_EUTILS}/efetch.fcgi", params={
                "db": "gene", "id": ids[0], "rettype": "gene_table", "retmode": "text"}, timeout=10)
            for line in r2.text.splitlines():
                if line.startswith("Also known as"):
                    [add(s) for s in re.split(r"[;,]", line.replace("Also known as","").strip())]
                    break
    except: pass
    # UniProt gene names + protein names
    try:
        r3 = requests.get("https://rest.uniprot.org/uniprotkb/search", params={
            "query": f"gene_exact:{gene_symbol} AND organism_id:9606 AND reviewed:true",
            "fields": "gene_names,protein_name", "format": "json", "size": 1}, timeout=10)
        results = r3.json().get("results", [])
        if results:
            entry = results[0]
            for g in entry.get("genes", []):
                [add(s.get("value","")) for s in g.get("synonyms",[])]
                [add(o.get("value","")) for o in g.get("orfNames",[])]
            pd = entry.get("proteinDescription", {})
            rec = pd.get("recommendedName", {})
            add(rec.get("fullName",{}).get("value",""))
            [add(sn.get("value","")) for sn in rec.get("shortNames",[])]
            for alt in pd.get("alternativeNames",[]):
                add(alt.get("fullName",{}).get("value",""))
                [add(sn.get("value","")) for sn in alt.get("shortNames",[])]
    except: pass
    return synonyms


def get_disease_synonyms(disease, efo_id=None):
    """疾患シノニムと MeSH heading を返す。"""
    synonyms = [disease]; seen = {disease.lower()}; mesh_heading = ""
    def add(name):
        name = str(name).strip()
        if name and name.lower() not in seen and len(name) >= 2:
            seen.add(name.lower()); synonyms.append(name)
    try:
        for q in [f"{disease}[MeSH Subheading]", disease]:
            r = requests.get(f"{_EUTILS}/esearch.fcgi", params={
                "db": "mesh", "term": q, "retmax": 1, "retmode": "json"}, timeout=10)
            ids = r.json().get("esearchresult", {}).get("idlist", [])
            if ids: break
        if ids:
            time.sleep(0.3)
            r2 = requests.get(f"{_EUTILS}/efetch.fcgi", params={"db":"mesh","id":ids[0]}, timeout=10)
            text = r2.text; lines = text.splitlines(); in_entry = False
            for line in lines:
                m = re.match(r"^\d+:\s+(.+)$", line.strip())
                if m and not mesh_heading:
                    mesh_heading = m.group(1).strip()
                    if mesh_heading.lower() != disease.lower(): add(mesh_heading)
            for line in lines:
                if line.strip().startswith("Entry Terms:"):
                    in_entry = True; continue
                if in_entry:
                    if not line.strip(): break
                    add(line.strip())
    except: pass
    if efo_id:
        try:
            if efo_id.startswith("EFO_"): onto="efo"; iri=f"http://www.ebi.ac.uk/efo/{efo_id}"
            elif efo_id.startswith("MONDO_"): onto="mondo"; iri=f"http://purl.obolibrary.org/obo/{efo_id.replace(':','_')}"
            else: iri=None
            if iri:
                r3 = requests.get(f"https://www.ebi.ac.uk/ols4/api/ontologies/{onto}/terms",
                                  params={"iri":iri}, timeout=12)
                tl = (r3.json().get("_embedded") or {}).get("terms",[])
                if tl:
                    t=tl[0]; add(t.get("label",""))
                    [add(s) for s in t.get("synonyms") or []]
        except: pass
    return synonyms, mesh_heading


def search_pubmed(gene, disease, max_results=10, disease_efo_id=None):
    """シノニム対応 PubMed 検索。"""
    gene_syns = get_gene_synonyms(gene)
    disease_syns, mesh_heading = get_disease_synonyms(disease, efo_id=disease_efo_id)
    print(f"    遺伝子シノニム ({len(gene_syns)}): {', '.join(gene_syns[:5])}")
    print(f"    疾患シノニム  ({len(disease_syns)}): {', '.join(disease_syns[:5])}")

    def _qt(term, field="Title/Abstract"):
        words = term.split()
        if len(words)<=1: return f'"{term}"[{field}]'
        return "(" + " AND ".join(f'"{w}"[{field}]' for w in words) + ")"
    def _qgs(n=8): return "(" + " OR ".join(_qt(s) for s in gene_syns[:n+1]) + ")"
    def _qdm(): return f'"{mesh_heading or disease}"[MeSH Terms]'
    def _qdt(n=4): return "(" + " OR ".join(_qt(s) for s in disease_syns[:n+1]) + ")"
    def _search(term):
        try:
            r = requests.get(f"{_EUTILS}/esearch.fcgi", params={
                "db":"pubmed","term":term,"retmax":max_results*3,"retmode":"json","sort":"pub+date"}, timeout=15)
            return r.json().get("esearchresult",{}).get("idlist",[])
        except: return []

    seen=set(); all_ids=[]
    for q in [
        f'"{gene}"[Gene/Protein Name] AND {_qdm()}',
        f'{_qt(gene)} AND {_qdm()}',
        f'{_qt(gene)} AND {_qdt(3)}',
        f'{_qgs(8)} AND {_qdm()}',
        f'{_qgs(8)} AND {_qdt(4)}',
    ]:
        for i in _search(q):
            if i not in seen: seen.add(i); all_ids.append(i)
        if len(all_ids) >= max_results*5: break
        time.sleep(0.2)
    if not all_ids: return []

    time.sleep(0.4)
    r = requests.post(f"{_EUTILS}/esummary.fcgi", data={
        "db":"pubmed","id":",".join(all_ids[:60]),"retmode":"json"}, timeout=20)
    result = r.json().get("result",{})
    papers = []
    for pmid in all_ids[:60]:
        if pmid not in result: continue
        item = result[pmid]
        papers.append({
            "pmid":pmid, "title":item.get("title",""), "journal":item.get("fulljournalname",""),
            "year":item.get("pubdate","")[:4], "authors":[a.get("name","") for a in item.get("authors",[])[:3]],
            "abstract":"", "relevance_score":0,
        })
    # アブストラクト & スコアリング
    if papers:
        time.sleep(0.3)
        try:
            rr = requests.get(f"{_EUTILS}/efetch.fcgi", params={
                "db":"pubmed","id":",".join(p["pmid"] for p in papers),
                "rettype":"abstract","retmode":"xml"}, timeout=30)
            amap = {pmid: re.sub(r"<[^>]+>"," ",txt).strip()
                    for pmid,txt in re.findall(
                        r"<PMID[^>]*>(\d+)</PMID>.*?<AbstractText[^>]*>(.*?)</AbstractText>",
                        rr.text, re.DOTALL)}
            glo = gene.lower(); dsyns_l=[s.lower() for s in disease_syns]
            dwds = [w for w in disease.lower().split() if len(w)>4]
            def dmatch(t):
                tl=t.lower(); return any(d in tl for d in dsyns_l) or all(w in tl for w in dwds)
            for p in papers:
                ab=amap.get(p["pmid"],""); p["abstract"]=ab
                tl=p["title"].lower(); al=ab.lower()
                gti=glo in tl; gab=glo in al
                dti=dmatch(p["title"]); dab=dmatch(ab)
                if gti and dti: p["relevance_score"]=4
                elif (gti or gab) and (dti or dab): p["relevance_score"]=3
                elif any(s.lower() in tl for s in gene_syns[1:]) and dti: p["relevance_score"]=2
                elif dti or dab: p["relevance_score"]=1
        except: pass
    papers.sort(key=lambda p:(p["relevance_score"],p["year"]),reverse=True)
    return papers[:max_results]

print("✓ PubMed 関数定義完了")

✓ PubMed 関数定義完了


In [5]:
# ============================================================
# 各種データコレクター
# ============================================================

# ── OpenTargets ────────────────────────────────────────────────────────────
_OT_API = "https://api.platform.opentargets.org/api/v4/graphql"

def _ot_search(q, entity):
    r = requests.post(_OT_API, json={"query": '''query($q:String!,$e:[String!]){search(queryString:$q,entityNames:$e,page:{index:0,size:5}){hits{id name entity}}}''',
                                     "variables":{"q":q,"e":[entity]}}, timeout=20)
    return r.json().get("data",{}).get("search",{}).get("hits",[])

def get_target_disease_evidence(gene, disease, gene_id=None, disease_id=None):
    if not disease_id:
        hits=[h for h in _ot_search(disease,"disease") if h.get("entity")=="disease"]
        if not hits: return {"error":f"Disease not found: {disease}"}
        disease_id=hits[0]["id"]
    if not gene_id:
        hits=[h for h in _ot_search(gene,"target") if h.get("entity")=="target"]
        exact=[h for h in hits if h.get("name","").upper()==gene.upper()]
        hits=exact or hits
        gene_id=hits[0]["id"] if hits else None
    target_data={}
    if gene_id:
        r=requests.post(_OT_API,json={"query":'''query($id:String!){target(ensemblId:$id){approvedName biotype functionDescriptions associatedDiseases(enableIndirect:true,page:{index:0,size:5}){rows{disease{name}score datatypeScores{id score}}} drugAndClinicalCandidates{rows{drug{id name maximumClinicalStage}maxClinicalStage diseases{disease{name}}}}}}''',
                                       "variables":{"id":gene_id}},timeout=25)
        target_data=r.json().get("data",{}).get("target") or {}
    assoc_score=None; dt_scores={}
    if gene_id:
        try:
            r2=requests.post(_OT_API,json={"query":'''query($efo:String!){disease(efoId:$efo){associatedTargets(enableIndirect:true,page:{index:0,size:500}){rows{target{id}score datatypeScores{id score}}}}}''',
                                            "variables":{"efo":disease_id}},timeout=30)
            for row in (r2.json().get("data",{}).get("disease") or {}).get("associatedTargets",{}).get("rows",[]):
                if (row.get("target") or {}).get("id")==gene_id:
                    assoc_score=row.get("score"); dt_scores={d["id"]:d["score"] for d in row.get("datatypeScores",[])}; break
        except: pass
    known_drugs=[]
    for row in ((target_data.get("drugAndClinicalCandidates") or {}).get("rows") or []):
        if not isinstance(row,dict): continue
        drug=row.get("drug") or {}
        diseases=[(d.get("disease") or {}).get("name","") for d in (row.get("diseases") or []) if isinstance(d,dict)]
        known_drugs.append({"drug":drug.get("name",""),"max_phase":row.get("maxClinicalStage"),"disease":diseases[0] if diseases else "","mechanism":""})
    return {"gene_symbol":gene,"ensembl_id":gene_id,"disease_id":disease_id,"association_score":assoc_score,
            "datatype_scores":dt_scores,"known_drugs":known_drugs}


# ── UniProt ────────────────────────────────────────────────────────────────
def get_protein_info(gene):
    r=requests.get("https://rest.uniprot.org/uniprotkb/search",params={
        "query":f"gene_exact:{gene} AND organism_id:9606 AND reviewed:true",
        "fields":"accession,gene_names,protein_name,organism_name,cc_function,cc_subcellular_location,cc_disease,go,keyword,length",
        "format":"json","size":1},timeout=15)
    results=r.json().get("results",[])
    if not results: return {"error":f"No UniProt entry for {gene}"}
    entry=results[0]; acc=entry.get("primaryAccession","")
    function_cc=""; subcell=[]; diseases=[]
    for comment in entry.get("comments",[]):
        ct=comment.get("commentType","")
        if ct=="FUNCTION":
            texts=comment.get("texts",[])
            if texts: function_cc=texts[0].get("value","")
        elif ct=="SUBCELLULAR LOCATION":
            [subcell.append(loc.get("location",{}).get("value","")) for loc in comment.get("subcellularLocations",[])]
    go_terms=[{"id":x.get("id",""),"term":next((p.get("value") for p in x.get("properties",[]) if p.get("key")=="GoTerm"),""),} for x in entry.get("uniProtKBCrossReferences",[]) if x.get("database")=="GO"]
    keywords=[kw.get("name","") for kw in entry.get("keywords",[])]
    protein_name=(entry.get("proteinDescription",{}).get("recommendedName",{}) or {}).get("fullName",{}).get("value","")
    return {"uniprot_id":acc,"gene":gene,"protein_name":protein_name,"function":function_cc,
            "subcellular_location":subcell,"go_terms":go_terms[:15],"keywords":keywords[:20]}


# ── GWAS + ClinVar ─────────────────────────────────────────────────────────
def get_gwas_associations(gene, disease_query=None):
    try:
        r=requests.get(f"https://www.ebi.ac.uk/gwas/rest/api/genes/{gene}/associations",
                       params={"projection":"associationByGene"},timeout=20)
        if r.status_code==404: return []
        assocs=r.json().get("_embedded",{}).get("associations",[])
        results=[]
        for a in assocs[:20]:
            trait=(a.get("efoTraits",[{}])[0].get("trait","") if a.get("efoTraits") else "")
            if disease_query and disease_query.lower() not in trait.lower(): continue
            study=a.get("study") or {}
            results.append({"trait":trait,"p_value":a.get("pvalue"),"or_beta":a.get("orPerCopyNum") or a.get("betaNum"),
                            "snps":[s.get("rsId","") for s in a.get("snps",[])],"study_id":study.get("accessionId",""),
                            "pub_date":study.get("publicationDate",""),"first_author":(study.get("author") or {}).get("fullname","")})
        return results
    except: return []

def get_clinvar_variants(gene):
    try:
        base="https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
        r=requests.get(f"{base}/esearch.fcgi",params={"db":"clinvar",
            "term":f"{gene}[Gene Name] AND (Pathogenic[Clinical significance] OR Likely pathogenic[Clinical significance])",
            "retmax":10,"retmode":"json"},timeout=15)
        ids=r.json().get("esearchresult",{}).get("idlist",[])
        if not ids: return []
        for _ in range(3):
            time.sleep(1)
            r2=requests.get(f"{base}/esummary.fcgi",params={"db":"clinvar","id":",".join(ids),"retmode":"json"},timeout=15)
            if r2.status_code!=429: break
        result=r2.json().get("result",{})
        return [{"variant_id":vid,"title":result[vid].get("title",""),
                 "clinical_significance":result[vid].get("clinical_significance",{}).get("description",""),
                 "condition":(result[vid].get("trait_set",[{}])[0].get("trait_name","") if result[vid].get("trait_set") else ""),
                 "uid":vid} for vid in ids if vid in result]
    except: return []


# ── ChEMBL ─────────────────────────────────────────────────────────────────
def get_drugs_for_target(gene):
    try:
        r=requests.get("https://www.ebi.ac.uk/chembl/api/data/target/search",
                       params={"q":gene,"format":"json","limit":5},timeout=20)
        human=[t for t in r.json().get("targets",[]) if t.get("organism")=="Homo sapiens"]
        if not human: return []
        tid=human[0].get("target_chembl_id")
        r2=requests.get("https://www.ebi.ac.uk/chembl/api/data/mechanism",
                        params={"target_chembl_id":tid,"format":"json","limit":20},timeout=20)
        drugs=[]; seen=set()
        for mech in r2.json().get("mechanisms",[]):
            mid=mech.get("molecule_chembl_id")
            if mid in seen: continue; seen.add(mid)
            try:
                r3=requests.get(f"https://www.ebi.ac.uk/chembl/api/data/molecule/{mid}",
                                params={"format":"json"},timeout=10)
                mol=r3.json()
                drugs.append({"chembl_id":mid,"name":mol.get("pref_name",mid),"max_phase":mol.get("max_phase"),
                              "mechanism":mech.get("mechanism_of_action","")})
            except: pass
        return drugs
    except: return []


# ── gnomAD ─────────────────────────────────────────────────────────────────
def get_constraint(gene):
    try:
        r=requests.post("https://gnomad.broadinstitute.org/api",json={"query":'''
            query G($g:String!){gene(gene_symbol:$g,reference_genome:GRCh38){gene_id gnomad_constraint{pLI oe_lof_upper lof_z oe_mis}}}''',
            "variables":{"g":gene}},timeout=20)
        c=(r.json().get("data",{}).get("gene") or {}).get("gnomad_constraint") or {}
        if not c: return {"error":"no data"}
        pli=c.get("pLI"); loeuf=c.get("oe_lof_upper")
        ess=("High (pLI≥0.9)" if (pli or 0)>=0.9 else "Moderate" if (pli or 0)>=0.5 else "Low")
        return {"pLI":pli,"LOEUF":loeuf,"lof_z":c.get("lof_z"),"oe_missense":c.get("oe_mis"),
                "essentiality":ess,"url":f"https://gnomad.broadinstitute.org/gene/{gene}?dataset=gnomad_r4"}
    except Exception as e: return {"error":str(e)}


# ── GTEx ───────────────────────────────────────────────────────────────────
_GTEX_KEY_TISSUES={"Heart_Left_Ventricle":"Heart","Liver":"Liver","Kidney_Cortex":"Kidney","Brain_Frontal_Cortex_Ba9":"Brain","Lung":"Lung","Muscle_Skeletal":"Skeletal muscle","Whole_Blood":"Whole blood"}

def get_tissue_expression(gene, top_n=10):
    try:
        r=requests.get("https://gtexportal.org/api/v2/reference/gene",
                       params={"geneSymbol":gene,"gencodeVersion":"v26","genomeBuild":"GRCh38/hg38"},timeout=15)
        genes=r.json().get("data",[])
        if not genes: return {"error":"not found"}
        vid=genes[0].get("gencodeId","")
        r2=requests.get("https://gtexportal.org/api/v2/expression/medianGeneExpression",
                        params={"gencodeId":vid,"datasetId":"gtex_v8"},timeout=20)
        records=r2.json().get("data",[])
        if not records: return {"error":"no data"}
        all_t=sorted([{"tissue":r.get("tissueSiteDetailId",""),"tpm":r.get("median",0)} for r in records],
                     key=lambda x:x["tpm"],reverse=True)
        tmap={r.get("tissueSiteDetailId",""):r.get("median",0) for r in records}
        key_t=[{"tissue":lb,"tpm":tmap.get(tid,0)} for tid,lb in _GTEX_KEY_TISSUES.items()]
        return {"top_tissues":all_t[:top_n],"key_tissues":sorted(key_t,key=lambda x:x["tpm"],reverse=True),
                "max_tissue":all_t[0]["tissue"] if all_t else "","max_tpm":all_t[0]["tpm"] if all_t else 0,
                "url":f"https://gtexportal.org/home/gene/{gene}"}
    except Exception as e: return {"error":str(e)}


# ── HPA ────────────────────────────────────────────────────────────────────
def get_expression_profile(gene):
    try:
        r=requests.get(f"https://www.proteinatlas.org/{gene}.json",timeout=20)
        if r.status_code==404: return {"error":"not found"}
        data=r.json()
        tissue_expr=[{"tissue":e.get("Tissue",""),"cell_type":e.get("Cell type",""),"level":e.get("Level",""),}
                     for e in (data.get("Normal tissue") or []) if e.get("Level") in ("High","Medium","Low")]
        subcell=list({loc.get("Location","") for loc in (data.get("Subcellular location") or []) if loc.get("Location")})
        protein_class=data.get("Protein class",[])
        high=[t for t in tissue_expr if t["level"]=="High"][:10] or tissue_expr[:10]
        return {"tissue_expression":high,"subcellular":subcell,
                "protein_class":protein_class if isinstance(protein_class,list) else [protein_class],
                "url":f"https://www.proteinatlas.org/{gene}"}
    except Exception as e: return {"error":str(e)}


# ── DGIdb ──────────────────────────────────────────────────────────────────
def get_dgidb_interactions(gene, max_results=20):
    try:
        r=requests.post("https://dgidb.org/api/graphql",json={"query":'''
            query($g:String!){genes(names:[$g]){nodes{interactions{drug{name approved}interactionScore interactionTypes{type}sources{sourceDbName}}}}}''',
            "variables":{"gene":gene}},timeout=20)
        nodes=(r.json().get("data",{}).get("genes",{}).get("nodes") or [])
        if not nodes: return []
        interactions=nodes[0].get("interactions") or []
        results=[]; seen=set()
        for ix in interactions:
            drug=ix.get("drug") or {}; name=(drug.get("name") or "").upper()
            if not name or name in seen: continue; seen.add(name)
            i_types=ix.get("interactionTypes") or []
            results.append({"drug_name":drug.get("name",""),"approved":drug.get("approved",False),
                            "interaction_type":(i_types[0].get("type","") if i_types else ""),"score":ix.get("interactionScore")})
        results.sort(key=lambda x:(x["approved"] is True,x["score"] or 0),reverse=True)
        return results[:max_results]
    except: return []


# ── ClinicalTrials ─────────────────────────────────────────────────────────
def get_trials(gene, disease, max_results=10):
    try:
        r=requests.get("https://clinicaltrials.gov/api/v2/studies",params={
            "query.cond":disease,"query.intr":gene,"pageSize":min(max_results*2,20),
            "format":"json","fields":"NCTId,BriefTitle,OverallStatus,Phase,StartDate,Condition,InterventionName"},timeout=20)
        results=[]
        for s in r.json().get("studies",[])[:max_results]:
            proto=s.get("protocolSection",{}); ident=proto.get("identificationModule",{})
            status=proto.get("statusModule",{}); design=proto.get("designModule",{})
            nct=ident.get("nctId",""); phases=design.get("phases",[])
            results.append({"nct_id":nct,"title":ident.get("briefTitle",""),"status":status.get("overallStatus",""),
                            "phase":"/".join(phases) if phases else "N/A","url":f"https://clinicaltrials.gov/study/{nct}"})
        return results
    except: return []


# ── AlphaFold ──────────────────────────────────────────────────────────────
def get_structure_info(gene):
    try:
        r0=requests.get("https://rest.uniprot.org/uniprotkb/search",params={
            "query":f"gene_exact:{gene} AND organism_id:9606 AND reviewed:true","fields":"accession","format":"json","size":1},timeout=10)
        uid=(r0.json().get("results",[{}])[0] or {}).get("primaryAccession","")
        if not uid: return {"error":"UniProt ID not found"}
        r=requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uid}",timeout=15)
        if r.status_code==404: return {"error":"not found"}
        entry=r.json()[0]; plddt=entry.get("meanPlddt")
        conf=("Very high (≥90)" if (plddt or 0)>=90 else "High (70-90)" if (plddt or 0)>=70 else "Low (<70)")
        return {"uniprot_id":uid,"entry_id":entry.get("entryId",""),"mean_plddt":plddt,"confidence":conf,
                "view_url":f"https://alphafold.ebi.ac.uk/entry/{uid}"}
    except Exception as e: return {"error":str(e)}


# ── Toxicity (PubChem + openFDA) ───────────────────────────────────────────
def assess_target_safety(gene, known_drugs):
    pb={}; ae={}
    try:
        r=requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/gene/genesymbol/{gene}/aids/JSON",timeout=15)
        if r.status_code!=404:
            aids=r.json().get("InformationList",{}).get("Information",[{}])[0].get("AID",[])
            pb={"assay_count":len(aids)}
    except: pass
    for drug in known_drugs[:3]:
        name=drug.get("drug") or drug.get("name","")
        if not name: continue
        try:
            r2=requests.get("https://api.fda.gov/drug/event.json",params={
                "search":f'patient.drug.medicinalproduct:"{name}"',
                "count":"patient.reaction.reactionmeddrapt.exact","limit":5},timeout=15)
            if r2.status_code not in (404,400):
                ae[name]=[{"reaction":x.get("term",""),"count":x.get("count",0)} for x in r2.json().get("results",[])]
        except: pass
    return {"pubchem_bioassay":pb,"drug_adverse_events":ae}

print("✓ 各種コレクター定義完了")

✓ 各種コレクター定義完了


In [6]:
# ============================================================
# PPIコレクター（IntAct / SIGNOR / Reactome） — ローカルキャッシュ付き
# ============================================================

# ── IntAct ─────────────────────────────────────────────────────────────────
_INTACT_CACHE = PPI_CACHE_DIR / "intact"

def _intact_cache_path(gene): return _INTACT_CACHE / f"{gene.upper()}.json"
def _intact_load(gene):
    p=_intact_cache_path(gene)
    if p.exists() and time.time()-p.stat().st_mtime < PPI_CACHE_TTL:
        return json.loads(p.read_text())
    return None
def _intact_save(gene, data):
    _INTACT_CACHE.mkdir(parents=True, exist_ok=True)
    _intact_cache_path(gene).write_text(json.dumps(data, ensure_ascii=False))

def get_intact_interactions(gene, max_results=20):
    cached=_intact_load(gene)
    if cached is not None:
        print("    [IntAct] キャッシュ使用"); return cached
    try:
        r=requests.get(f"https://www.ebi.ac.uk/intact/ws/interaction/findInteractions/{gene}",
                       params={"page":0,"pageSize":max_results,"query":"species:9606"},timeout=20)
        if r.status_code==404: _intact_save(gene,[]); return []
        r.raise_for_status(); data=r.json()
        interactions=[]
        for item in data.get("content",[]):
            parts=item.get("participants",[])
            names=[p.get("preferredName","") or str(p) for p in parts if isinstance(p,dict)] or [str(p) for p in parts]
            partners=[n for n in names if gene.upper() not in n.upper()]
            interactions.append({"partners":partners,"confidence":item.get("intactScore"),"interaction_id":item.get("interactionAc","")})
        _intact_save(gene, interactions)
        return interactions
    except Exception as e: print(f"    [IntAct] エラー: {e}"); return []


# ── SIGNOR ─────────────────────────────────────────────────────────────────
_SIGNOR_CACHE_FILE = PPI_CACHE_DIR / "signor_9606.tsv"

def _get_signor_tsv():
    PPI_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if _SIGNOR_CACHE_FILE.exists() and time.time()-_SIGNOR_CACHE_FILE.stat().st_mtime < SIGNOR_CACHE_TTL:
        return _SIGNOR_CACHE_FILE.read_text(encoding="utf-8")
    print("    [SIGNOR] TSV ダウンロード中（初回 or 7日経過）...")
    r=requests.get("https://signor.uniroma2.it/getData.php?organism=9606&format=tsv",timeout=60)
    r.raise_for_status()
    _SIGNOR_CACHE_FILE.write_text(r.text, encoding="utf-8")
    return r.text

def get_signor_interactions(gene):
    text=_get_signor_tsv(); gene_upper=gene.upper(); results=[]
    for line in text.splitlines():
        parts=line.split("\t")
        if len(parts)<9: continue
        ea=parts[0].strip().upper(); eb=parts[4].strip().upper()
        if ea!=gene_upper and eb!=gene_upper: continue
        if parts[1].strip() not in ("protein","complex") and parts[5].strip() not in ("protein","complex"): continue
        partner=parts[4].strip() if ea==gene_upper else parts[0].strip()
        try: score=float(parts[23]) if len(parts)>23 and parts[23] else None
        except: score=None
        results.append({
            "source":gene if ea==gene_upper else partner,
            "target":partner if ea==gene_upper else gene,
            "effect":parts[8].strip(),"mechanism":(parts[9].strip() if len(parts)>9 else ""),
            "score":score,"db":"SIGNOR"})
    return results


# ── Reactome interactions ──────────────────────────────────────────────────
_REACTOME_PPI_CACHE = PPI_CACHE_DIR / "reactome"

def _reactome_cache_path(gene): return _REACTOME_PPI_CACHE / f"{gene.upper()}.json"
def _reactome_load(gene):
    p=_reactome_cache_path(gene)
    if p.exists() and time.time()-p.stat().st_mtime < PPI_CACHE_TTL:
        return json.loads(p.read_text())
    return None
def _reactome_save(gene, data):
    _REACTOME_PPI_CACHE.mkdir(parents=True, exist_ok=True)
    _reactome_cache_path(gene).write_text(json.dumps(data, ensure_ascii=False))

def get_reactome_interactions(gene, max_results=50):
    cached=_reactome_load(gene)
    if cached is not None:
        print("    [Reactome] キャッシュ使用"); return cached
    try:
        r0=requests.get("https://rest.uniprot.org/uniprotkb/search",params={
            "query":f"gene_exact:{gene} AND organism_id:9606 AND reviewed:true","fields":"accession","format":"json","size":1},timeout=10)
        uid=(r0.json().get("results",[{}])[0] or {}).get("primaryAccession","")
        if not uid: _reactome_save(gene,[]); return []
        r=requests.get(f"https://reactome.org/ContentService/data/interactors/static/proteins/{uid}",
                       params={"page":1,"pageSize":max_results},timeout=20)
        if r.status_code==404: _reactome_save(gene,[]); return []
        center=gene.upper()
        interactions=[]
        for entry in r.json().get("entities",[]):
            for ix in entry.get("interactors",[]):
                partner=(ix.get("geneName") or ix.get("acc") or "").strip().upper()
                if not partner or partner==center: continue
                interactions.append({"source":center,"target":partner,"effect":"","mechanism":"Reactome"})
        _reactome_save(gene, interactions)
        return interactions
    except Exception as e: print(f"    [Reactome] エラー: {e}"); return []

print("✓ PPI コレクター定義完了")

✓ PPI コレクター定義完了


In [7]:
# ============================================================
# collect_all — 全コレクターを並列実行
# ============================================================
MAX_RETRIES = 3
RETRY_WAIT  = [2, 5, 10]

def _run_retry(fn, key, log):
    last=None
    for attempt in range(MAX_RETRIES):
        try:
            result=fn()
            if attempt>0: log(f"{key}: OK (retry {attempt})")
            else: log(f"{key}: OK")
            return result, None
        except Exception as e:
            last=e
            w=RETRY_WAIT[min(attempt,2)]
            log(f"{key}: error ({attempt+1}/{MAX_RETRIES}) — {e}  →{w}s")
            if attempt<MAX_RETRIES-1: time.sleep(w)
    return None, str(last)


def collect_all(gene, disease, verbose=True, disease_id=None, gene_id=None):
    def log(msg):
        if verbose: print(f"  [+] {msg}")
    tasks = {
        "pubmed":         lambda: search_pubmed(gene, disease, max_results=8, disease_efo_id=disease_id),
        "opentargets":    lambda: get_target_disease_evidence(gene, disease, gene_id=gene_id, disease_id=disease_id),
        "uniprot":        lambda: get_protein_info(gene),
        "intact":         lambda: get_intact_interactions(gene, max_results=15),
        "gwas":           lambda: get_gwas_associations(gene, disease),
        "clinvar":        lambda: get_clinvar_variants(gene),
        "chembl":         lambda: get_drugs_for_target(gene),
        "gnomad":         lambda: get_constraint(gene),
        "gtex":           lambda: get_tissue_expression(gene),
        "hpa":            lambda: get_expression_profile(gene),
        "dgidb":          lambda: get_dgidb_interactions(gene),
        "clinicaltrials": lambda: get_trials(gene, disease),
        "alphafold":      lambda: get_structure_info(gene),
        "reactome":       lambda: [], # パスウェイは PPI とは別途呼び出し
    }
    results={}; errors={}
    def _task(key,fn): return key, *_run_retry(fn, key, log)
    with ThreadPoolExecutor(max_workers=10) as ex:
        futs={ex.submit(_task,k,fn):k for k,fn in tasks.items()}
        for f in as_completed(futs):
            key,result,err=f.result()
            results[key]=result
            if err: errors[key]=err
    # toxicity
    known_drugs=[]
    if results.get("chembl"): known_drugs.extend(results["chembl"])
    if isinstance(results.get("opentargets"),dict): known_drugs.extend(results["opentargets"].get("known_drugs",[]))
    results["toxicity"],terr=_run_retry(lambda:assess_target_safety(gene,known_drugs),"toxicity",log)
    if terr: errors["toxicity"]=terr
    if verbose:
        print(f"\n  完了: {sum(1 for k in tasks if k not in errors and results.get(k) is not None)}/{len(tasks)} ソース")
    return {"gene":gene,"disease":disease,"evidence":results,"collection_errors":errors}


# ============================================================
# build_llm_context — LLM用コンテキスト整形
# ============================================================
def _trunc(text, max_chars):
    if not text or len(text)<=max_chars: return text
    w=text[:max_chars]
    for sep in ('. ','!\n','? '):
        idx=w.rfind(sep)
        if idx>=int(max_chars*0.55): return w[:idx+1]
    return w[:w.rfind(' ',0,max_chars)] + ' ...'


def build_llm_context(aggregated, config=None):
    cfg={**CONTEXT_CONFIG,**(config or {})}
    gene=aggregated["gene"]; disease=aggregated["disease"]; ev=aggregated["evidence"]
    sections=[f"# Evidence: {gene} × {disease}\n(Cite inline: [Paper 1], [UniProt], [GWAS 1], etc.)\n"]
    ref_reg={"paper":[],"disease":[],"gene":[],"drug":[]}; ref_cnt={k:0 for k in ref_reg}
    def add_ref(cat,prefix,short,full):
        if prefix: ref_cnt[cat]+=1; tag=f"[{prefix} {ref_cnt[cat]}]"
        else: tag=f"[{cat}]"
        ref_reg[cat].append((tag,short,full)); return tag

    # UniProt
    uni=ev.get("uniprot") or {}
    if uni and "error" not in uni:
        uid=uni.get("uniprot_id",""); url=f"https://www.uniprot.org/uniprotkb/{uid}"
        ref=add_ref("gene","UniProt",f"UniProt {gene} ({uid}). {url}",f"UniProt. {url}")
        func=_trunc(uni.get("function") or "",cfg["uniprot_chars"])
        kws=", ".join(uni.get("keywords",[])[:cfg["uniprot_keywords"]]) or "N/A"
        gos="; ".join(g["term"] for g in uni.get("go_terms",[])[:cfg["uniprot_go_terms"]]) or "N/A"
        sections.append(f"## Gene/Protein {ref}\n- Name: {uni.get('protein_name','N/A')}\n- Function: {func}\n- Location: {', '.join(uni.get('subcellular_location',[])[:4]) or 'N/A'}\n- Keywords: {kws}\n- GO: {gos}\n")
    # OpenTargets
    ot=ev.get("opentargets") or {}
    if ot and "error" not in ot:
        score=ot.get("association_score"); ss=f"{score:.3f}" if score else "N/A"
        dt=" | ".join(f"{k}:{v:.2f}" for k,v in (ot.get("datatype_scores") or {}).items())
        ref=add_ref("disease","OpenTargets",f"OT {gene}×{disease}","OpenTargets Platform")
        sections.append(f"## OpenTargets {ref}\n- Score: {ss} | {dt}\n")
    # GWAS
    gh=ev.get("gwas") or []
    if gh:
        lines=[]
        for h in gh[:cfg["max_gwas"]]:
            ref=add_ref("disease","GWAS",f"{h.get('first_author','')} GWAS {h.get('study_id','')}","GWAS Catalog")
            lines.append(f"  - {h['trait']} p={h['p_value']} OR={h['or_beta']} {ref}")
        sections.append("## GWAS\n"+"\n".join(lines)+"\n")
    # ClinVar
    cv=ev.get("clinvar") or []
    if cv:
        lines=[]
        for v in cv[:cfg["max_clinvar"]]:
            vid=str(v.get("uid","") or v.get("variant_id",""))
            url=f"https://www.ncbi.nlm.nih.gov/clinvar/variation/{vid}/"
            ref=add_ref("disease","ClinVar",f"ClinVar {vid}. {url}","ClinVar. NCBI.")
            lines.append(f"  - {v['title'][:65]} | {v['clinical_significance']} | {v['condition']} {ref}")
        sections.append("## ClinVar\n"+"\n".join(lines)+"\n")
    # Drugs
    drugs_chembl=ev.get("chembl") or []
    ot_drugs=(ot.get("known_drugs",[]) if ot else [])
    all_drugs={d.get("name") or d.get("drug",""):d for d in drugs_chembl+ot_drugs if d.get("name") or d.get("drug")}
    if all_drugs:
        lines=[]
        for name,d in list(all_drugs.items())[:cfg["max_drugs"]]:
            ref=add_ref("drug","ChEMBL",f"{name}","ChEMBL")
            lines.append(f"  - {name} | Ph:{d.get('max_phase')} | {(d.get('mechanism') or '')[:60]} {ref}")
        sections.append(f"## Drugs targeting {gene}\n"+"\n".join(lines)+"\n")
    # IntAct
    interactions=ev.get("intact") or []
    if interactions:
        partners=list(dict.fromkeys(p for ix in interactions[:cfg["max_interactions"]] for p in ix.get("partners",[])))[:cfg["max_interactions"]]
        ref=add_ref("gene","IntAct",f"IntAct PPI {gene}","IntAct. EMBL-EBI")
        sections.append(f"## PPI (IntAct) {ref}\n- Interactors: {', '.join(partners) or 'N/A'}\n")
    # Safety
    tox=ev.get("toxicity") or {}
    pb=tox.get("pubchem_bioassay",{}); ae=tox.get("drug_adverse_events",{})
    ref_pc=add_ref("gene","PubChem",f"PubChem {gene}","PubChem BioAssay. NIH.")
    ae_str="; ".join(f"{drug}: "+", ".join(f"{e['reaction']}({e['count']})" for e in evts[:2]) for drug,evts in list(ae.items())[:2]) or "N/A"
    sections.append(f"## Safety {ref_pc}\n- BioAssays: {pb.get('assay_count',0)} | AEs: {ae_str}\n")
    # Literature
    papers=ev.get("pubmed") or []
    if papers:
        blks=[]
        for p in papers[:cfg["max_papers"]]:
            url=f"https://pubmed.ncbi.nlm.nih.gov/{p.get('pmid','')}/"
            ref=add_ref("paper","Paper",f"{p.get('authors',[['?']])[0]} {p.get('journal','')} {p.get('year','')} {url}","")
            snippet=_trunc(p.get("abstract") or "",cfg["abstract_chars"]) or "(no abstract)"
            blks.append(f"### {ref} {p['title'][:80]} ({p['year']})\n_{', '.join(p.get('authors',[])[:2])} | {p.get('journal','')}_\n{snippet}\n")
        sections.append("## Literature\n\n"+"\n".join(blks))
    # gnomAD
    gnom=ev.get("gnomad") or {}
    if gnom and "error" not in gnom:
        ref=add_ref("gene","gnomAD",f"gnomAD {gene}","gnomAD. Broad Institute.")
        sections.append(f"## Constraint (gnomAD) {ref}\n- pLI:{gnom.get('pLI')} LOEUF:{gnom.get('LOEUF')} — {gnom.get('essentiality')}\n")
    # GTEx
    gtex_data=ev.get("gtex") or {}
    if gtex_data and "error" not in gtex_data:
        ref=add_ref("gene","GTEx",f"GTEx {gene}","GTEx. 2020.")
        top3=", ".join(f"{t['tissue']}({t['tpm']:.0f})" for t in gtex_data.get("top_tissues",[])[:cfg["gtex_top_n"]])
        sections.append(f"## Expression GTEx {ref}\n- Top: {top3}\n")
    # HPA
    hpa_data=ev.get("hpa") or {}
    if hpa_data and "error" not in hpa_data:
        ref=add_ref("gene","HPA",f"HPA {gene}","Human Protein Atlas. 2015.")
        subcell=", ".join(hpa_data.get("subcellular",[])[:4]) or "N/A"
        tissues=" | ".join(f"{t['tissue']}:{t['level']}" for t in hpa_data.get("tissue_expression",[])[:cfg["hpa_top_n"]]) or "N/A"
        sections.append(f"## Protein Atlas {ref}\n- Location:{subcell}\n- Expression: {tissues}\n")
    # DGIdb
    dgi=ev.get("dgidb") or []
    if dgi:
        ref=add_ref("drug","DGIdb",f"DGIdb {gene}","DGIdb. 2024.")
        rows=" | ".join(f"{d['drug_name']}({'✓' if d.get('approved') else 'inv'})" for d in dgi[:cfg["max_dgidb"]])
        sections.append(f"## DGIdb {ref}\n- {rows}\n")
    # ClinicalTrials
    ct=ev.get("clinicaltrials") or []
    if ct:
        ref=add_ref("disease","ClinicalTrials",f"ClinicalTrials {gene}×{disease}","ClinicalTrials.gov")
        rows=" | ".join(f"{t['nct_id']}({t['phase']},{t['status'][:8]})" for t in ct[:cfg["max_trials"]])
        sections.append(f"## Clinical Trials {ref}\n- {rows}\n")
    # AlphaFold
    af=ev.get("alphafold") or {}
    if af and "error" not in af:
        ref=add_ref("gene","AlphaFold",f"AlphaFold {af.get('entry_id','')}","AlphaFold DB. EMBL-EBI.")
        sections.append(f"## Structure AlphaFold {ref}\n- pLDDT:{af.get('mean_plddt')} — {af.get('confidence')}\n")
    # References
    ref_lines=["## References\n"]
    for cat,header in [("paper","### Papers"),("disease","### Disease DBs"),("gene","### Gene DBs"),("drug","### Drug DBs")]:
        entries=ref_reg.get(cat,[])
        if entries:
            ref_lines.append(header)
            for tag,short,_ in entries: ref_lines.append(f"{tag} {short}")
            ref_lines.append("")
    sections.append("\n".join(ref_lines))
    return "\n".join(sections)

print("✓ collect_all + build_llm_context 定義完了")

✓ collect_all + build_llm_context 定義完了


In [8]:
# ============================================================
# PPI ネットワーク構築 + エンリッチメント + 可視化
# ============================================================
_DB_COLORS = {"IntAct":"#4ECDC4","SIGNOR":"#45B7D1","Reactome":"#FFB347","multi":"#8E44AD"}

def _db_color(db): return _DB_COLORS.get(db, "#DDD")

def build_ppi_network(gene, use_reactome=True):
    if not HAS_NX: return None
    G=nx.Graph(); center=gene.upper()
    G.add_node(center, color="#FF6B6B", size=25, db="center")
    def _item_partners(item, source):
        score=item.get("score",item.get("confidence"))
        try: score=float(score) if score is not None else None
        except: score=None
        out=[]
        src=(item.get("source") or "").strip().upper(); tgt=(item.get("target") or "").strip().upper()
        if src and tgt:
            partner=tgt if src==center else src
            if partner and partner!=center: out.append((partner,score))
        for p in item.get("partners",[]) or []:
            p=str(p).strip().upper()
            if p and p!=center: out.append((p,score))
        return out
    def add_edges(interactions, src_label):
        for item in interactions:
            for partner,score in _item_partners(item, src_label):
                if partner not in G:
                    G.add_node(partner, color=_db_color(src_label), size=15, db=src_label)
                elif src_label not in G.nodes[partner].get("db",""):
                    G.nodes[partner]["db"] += f",{src_label}"
                if G.has_edge(center, partner):
                    ed=G.edges[center,partner]
                    ed["weight"]=ed.get("weight",1)+1
                    ed.setdefault("dbs",set()).add(src_label)
                    if score is not None: ed["score"]=score if ed.get("score") is None else max(ed["score"],score)
                else:
                    G.add_edge(center, partner, weight=1, db=src_label, dbs={src_label}, score=score)
    print("  IntAct 取得中...")
    add_edges(get_intact_interactions(gene), "IntAct")
    print("  SIGNOR 取得中...")
    add_edges(get_signor_interactions(gene), "SIGNOR")
    if use_reactome:
        print("  Reactome 取得中...")
        add_edges(get_reactome_interactions(gene), "Reactome")
    print(f"  ネットワーク: {G.number_of_nodes()} nodes / {G.number_of_edges()} edges")
    return G


def rank_partners(G, center, max_n=30):
    def key(n):
        ed=G.edges[center,n]; dbs=ed.get("dbs") or {d for d in (ed.get("db","") or "").split(",") if d}
        score=ed.get("score"); score=score if score is not None else -1.0
        return (len(dbs), score, ed.get("weight",1))
    return sorted(G.neighbors(center), key=key, reverse=True)[:max_n]


def run_network_enrichment(G, top_n=30):
    if G is None: return {}
    gene_list=[n for n in G.nodes if n]
    print(f"  エンリッチメント対象: {len(gene_list)} 遺伝子")
    try:
        r=requests.post("https://biit.cs.ut.ee/gprofiler/api/gost/profile/",json={
            "organism":"hsapiens","query":gene_list,
            "sources":["GO:BP","GO:MF","GO:CC","REAC","WP"],
            "user_threshold":0.05,"significance_threshold_method":"fdr","no_evidences":True},timeout=30)
        raw=r.json().get("result") or []
        results=[{"source":i.get("source",""),"term_id":i.get("native",""),"term_name":i.get("name",""),
                  "p_value":i.get("p_value",1.0),"intersection_size":i.get("intersection_size",0),
                  "genes":i.get("intersections",[])} for i in raw[:top_n]]
        results.sort(key=lambda x:x["p_value"])
        print(f"  エンリッチメント: {len(results)} 有意項目")
        return {"gene_list":gene_list,"results":results}
    except Exception as e:
        print(f"  エンリッチメントエラー: {e}"); return {}


def render_ppi_image(G, gene, out_path, enrichment=None, max_nodes=30, dpi=130):
    if G is None or not HAS_NX or not HAS_MPL: return None
    center=gene.upper()
    if center not in G: return None
    neighbors=rank_partners(G, center, max_n=max_nodes)
    if not neighbors: return None
    pos={center:(0.5,1.06)}
    n=len(neighbors); cols=max(1,math.ceil(math.sqrt(n*1.8))); rows=math.ceil(n/cols)
    for i,node in enumerate(neighbors):
        r,c=divmod(i,cols); x=(c+0.5)/cols; jitter=0.04*(1 if i%2==0 else -1)
        pos[node]=(x, 0.55-(r/max(1,rows))*0.55+jitter)
    def node_color(node):
        ed=G.edges[center,node]; dbs=ed.get("dbs") or {d for d in (ed.get("db","") or "").split(",") if d}
        return _DB_COLORS["multi"] if len(dbs)>=2 else _db_color(next(iter(dbs)) if dbs else "")
    fig,ax=plt.subplots(figsize=(8.5,6.0)); ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.15,1.20); ax.axis("off")
    cx,cy=pos[center]
    for node in neighbors:
        nx_,ny_=pos[node]; ax.plot([cx,nx_],[cy,ny_],color="#C8C8C8",lw=0.8,zorder=1)
    used_dbs=set(); has_multi=False
    for node in neighbors:
        x,y=pos[node]; ed=G.edges[center,node]; dbs=ed.get("dbs") or {d for d in (ed.get("db","") or "").split(",") if d}
        if len(dbs)>=2: has_multi=True
        else: used_dbs|=dbs
        ax.scatter([x],[y],s=320,c=node_color(node),edgecolors="#222",linewidths=0.8,zorder=2)
        ax.text(x,y-0.055,node,ha="center",va="top",fontsize=7.5,color="#222",zorder=3)
    ax.scatter([cx],[cy],s=900,c="#FF6B6B",marker="*",edgecolors="#222",linewidths=1.0,zorder=4)
    ax.text(cx,cy+0.06,center,ha="center",va="bottom",fontsize=12,fontweight="bold",color="#B22222",zorder=5)
    handles=[Line2D([0],[0],marker="*",color="w",label=f"{gene} (target)",markerfacecolor="#FF6B6B",markeredgecolor="#222",markersize=15)]
    for db in ("IntAct","SIGNOR","Reactome"):
        if db in used_dbs: handles.append(Line2D([0],[0],marker="o",color="w",label=db,markerfacecolor=_db_color(db),markeredgecolor="#222",markersize=9))
    if has_multi: handles.append(Line2D([0],[0],marker="o",color="w",label="multiple DBs",markerfacecolor=_DB_COLORS["multi"],markeredgecolor="#222",markersize=9))
    ax.legend(handles=handles,loc="upper right",fontsize=8,frameon=True,framealpha=0.9,edgecolor="#CCC")
    ax.set_title(f"PPI Network — {gene}  ({G.number_of_nodes()} nodes; top {len(neighbors)})",fontsize=11)
    os.makedirs(os.path.dirname(out_path) or ".",exist_ok=True)
    fig.savefig(out_path,dpi=dpi,bbox_inches="tight",facecolor="white"); plt.close(fig)
    return out_path


def network_summary_for_llm(G, gene, enrichment, max_partners=10, max_terms=15):
    if G is None: return ""
    center=gene.upper(); partners=rank_partners(G, center, max_n=max_partners)
    lines=[f"## PPI Network ({gene})",f"- Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}",
           f"- Key interactors: {', '.join(partners)}"]
    for r in (enrichment or {}).get("results",[])[:max_terms]:
        lines.append(f"- [{r['source']}] {r['term_name']} ({r.get('term_id','')}) p={r['p_value']:.2e}")
    return "\n".join(lines)

print("✓ PPI / エンリッチメント 定義完了")

✓ PPI / エンリッチメント 定義完了


In [9]:
# ============================================================
# 仮説生成プロンプト
# ============================================================
HYPOTHESIS_PROMPT_EN = """You are a drug discovery scientist. Write a hypothesis report for {gene} in {disease} using ONLY the evidence below.

STRICT RULES:
1. Only use reference tags that ACTUALLY APPEAR in the context (e.g. [Paper 1], [UniProt 1]). Never invent a tag.
2. If data is absent, write "No data available" — never fabricate.
3. Use specific values (p-values, TPM, pLI scores, drug names, etc.).
4. Cite every factual claim with a tag.

--- EVIDENCE ---
{context}
--- END EVIDENCE ---

## 1. Target Validity

### 1a. Genetic Evidence
State GWAS hits and ClinVar variants with specific traits, p-values, and citations.
Confidence: Low / Moderate / High / Very High — one sentence reason.

### 1b. Functional Evidence
Summarize literature findings about {gene} in {disease} with specific citations.
Confidence: Low / Moderate / High / Very High — one sentence reason.

### 1c. Clinical Relevance
List existing drugs targeting {gene} with phase and mechanism.

### 1d. Expression & Network
Top expressing tissues (GTEx TPM), HPA subcellular location, key PPI partners and pathways.

### 1e. Overall Validity Score
**Overall: Low / Moderate / High / Very High**
Two-sentence synthesis.

---

## 2. Molecular Mechanism
Describe the molecular mechanism linking {gene} to {disease}:
- **Protein function:** what does {gene} normally do at the molecular level?
- **Dysregulation in disease:** how is {gene} expression/activity/structure altered in {disease}? (overexpressed / loss-of-function / mutation / mislocalised)
- **Direct molecular effects:** what immediate molecular events result (kinase activity change, PPI disruption, transcription factor binding)?
- **Key interactors:** which PPI partners or pathway nodes are most affected? Cite network data.
- **Downstream signalling cascade:** step-by-step from {gene} perturbation → effector proteins → cellular phenotype → tissue pathology → {disease}.
- **Supporting evidence:** cite specific findings from literature, GWAS, ClinVar, and pathway enrichment.

---

## 3. Disease Mechanism
Step-by-step narrative: how does {gene} dysfunction lead to the tissue-level and clinical pathology of {disease}? Connect molecular events to organ function and patient symptoms. Use specific pathway names, effectors, and cite literature.

---

## 4. Therapeutic Hypothesis
### 4a. Treatment Hypothesis (bullet points)
- **Intervention:** proposed modulation (inhibit/activate/degrade/replace)
- **Target effect:** change in {gene} activity
- **Downstream effect:** mechanism correction
- **Clinical outcome:** expected improvement
- **Key evidence:** strongest supporting data [citations]
- **Falsifiable prediction:** one testable statement

### 4b. One-sentence hypothesis
"If [intervention on {gene}] then [outcome in {disease}] because [mechanism]."

---

## 5. Modality
Best modality (small molecule/antibody/PROTAC/ASO/gene therapy):
- Rationale from location and druggability
- AlphaFold pLDDT and structural confidence
- Key advantage over alternatives

---

## 6. Existing Drug Landscape
Known drugs/candidates with phases. Repositioning opportunities.

---

## 7. Safety Assessment
- On-target risks from gene function
- Expression in safety-relevant tissues (heart/liver/kidney/CNS) with TPM
- gnomAD pLI/LOEUF interpretation
- Off-target toxicity signals

---

## 8. Recommended Experiments
| Experiment | Endpoint | Expected Result |
|---|---|---|
| (3–5 specific experiments) | | |

---

## 9. Key Uncertainties
- Evidence gaps
- Alternative hypotheses
- Major risks
"""

HYPOTHESIS_PROMPT_JA = """あなたは創薬の専門家です。以下のエビデンスのみを使って、{gene}を標的とした{disease}の創薬仮説レポートを日本語で作成してください。

【絶対ルール】
1. コンテキスト内に実際に登場する引用タグのみ使うこと。存在しないタグを作らない。
2. データがない項目は「データなし」と書く。情報を推測・捏造しない。
3. コンテキストの具体的な数値を使う（p値・TPM・pLIスコア・薬剤名など）。
4. 事実的な主張には必ずタグを引用する。

--- エビデンス ---
{context}
--- エビデンスここまで ---

## 1. ターゲット妥当性評価
### 1a. 遺伝的エビデンス
GWASヒット数・ClinVar変異数を明記。具体的な形質名・p値を引用付きで。
信頼度：低 / 中 / 高 / 非常に高 — 理由を一文で。

### 1b. 機能的エビデンス
文献が{gene}と{disease}について何を述べているかを要約（引用付き）。
信頼度：低 / 中 / 高 / 非常に高 — 理由を一文で。

### 1c. 臨床的関連性
{gene}標的の既存薬を列挙（薬剤名・フェーズ・作用機序）。

### 1d. 発現プロファイル・ネットワーク
GTEx上位発現組織（TPM値）・HPA細胞内局在・主要PPI相互作用パートナーを記述。

### 1e. 総合スコア
**総合：低 / 中 / 高 / 非常に高**
2文以内で根拠をまとめる。

---

## 2. 分子メカニズム
{gene}と{disease}をつなぐ分子メカニズムを詳述：
- **タンパク質の正常機能：** {gene}は分子レベルで何をしているか？
- **疾患における機能異常：** {disease}において{gene}はどう変化しているか？（過剰発現/機能喪失/変異/局在異常）
- **直接的な分子イベント：** どのような即時の分子イベントが起こるか？
- **主要な相互作用パートナー：** どのPPIパートナーやパスウェイノードが最も影響を受けるか？ネットワークデータを引用。
- **下流シグナル伝達カスケード：** {gene}変化 → エフェクター → 細胞表現型 → 組織病理 → {disease}発症のステップを記述。
- **支持エビデンス：** 文献・GWAS・ClinVar・エンリッチメントの具体的知見を引用。

---

## 3. 疾患メカニズムの考察
上記の分子イベントが組織・臓器レベルの病態と患者症状にどうつながるかを記述。{gene}機能異常から{disease}の臨床症状に至るステップをパスウェイ名・エフェクター・文献引用とともに説明。

---

## 4. 治療仮説
### 4a. 疾患治療仮説（箇条書き）
- **介入方法：** 具体的介入（阻害/活性化/分解/補充）
- **ターゲットへの効果：** 介入による{gene}活性の変化
- **下流効果：** 分子メカニズム異常をどう是正するか
- **期待される臨床効果：** {disease}への改善
- **支持エビデンス：** 最も強いエビデンスを引用タグで
- **検証可能な予測：** 反証可能な1つの予測

### 4b. 仮説一文
「{gene}に対して〔介入〕を行うと、{disease}患者において〔臨床効果〕が得られる。これは〔分子メカニズム〕による。」

---

## 5. モダリティ提案
---

## 6. 既存薬景観とリポジショニング
---

## 7. 安全性・毒性リスク評価
---

## 8. 推奨次期実験
| 実験種別 | エンドポイント | 期待される結果 |
|---|---|---|

---

## 9. 主要な不確実性・限界
"""

def generate_hypothesis(gene, disease, context, llm_client, temperature=0.3, lang="en", stream_callback=None):
    template = HYPOTHESIS_PROMPT_JA if lang=="ja" else HYPOTHESIS_PROMPT_EN
    prompt = template.format(gene=gene, disease=disease, context=context)
    kwargs = dict(temperature=temperature, max_tokens=4000)
    if stream_callback: kwargs["stream_callback"] = stream_callback
    return llm_client.generate(prompt, **kwargs)

print("✓ 仮説生成プロンプト定義完了")

✓ 仮説生成プロンプト定義完了


In [10]:
# ============================================================
# レポート整形 + パイプライン
# ============================================================
_SRC_LABEL = {"GO:BP":"GO Biological Process","GO:MF":"GO Molecular Function",
              "GO:CC":"GO Cellular Component","REAC":"Reactome","WP":"WikiPathways",
              "HP":"Human Phenotype","CORUM":"Protein Complexes"}
_SRC_ORDER = ["GO:BP","GO:MF","GO:CC","REAC","WP","HP","CORUM"]

def enrichment_md(enrichment, top_per_source=5):
    results=(enrichment or {}).get("results",[])
    if not results: return ""
    by_src=defaultdict(list)
    for r in results: by_src[r["source"]].append(r)
    ordered=[(s,by_src[s]) for s in _SRC_ORDER if s in by_src]
    ordered+=[(s,by_src[s]) for s in sorted(by_src) if s not in _SRC_ORDER]
    lines=["## Functional Enrichment (g:Profiler, FDR < 0.05)",""]
    for src,terms in ordered:
        lines.append(f"### {_SRC_LABEL.get(src,src)}"); lines.append("")
        lines.append("| Term | p-value | Genes (overlap) |"); lines.append("|---|:---:|---|")
        for t in terms[:top_per_source]:
            genes=", ".join(t.get("genes",[])[:8])
            lines.append(f"| {t['term_name'][:70]} `{t.get('term_id','')}` | {t['p_value']:.2e} | {genes} ({t.get('intersection_size',0)}) |")
        lines.append("")
    lines.append(f"<sub><sup>有意項目合計: {len(results)} 件 (FDR<0.05) — [g:Profiler](https://biit.cs.ut.ee/gprofiler/gost)</sup></sub>\n")
    return "\n".join(lines)

def ppi_md(gene, image_filename, partners=None):
    lines = ["## PPI Network", "", f"![PPI network of {gene}]({image_filename})", "",
             f"<sub><sup>★ = {gene} (target) ／ 色 = データソース（IntAct/SIGNOR/Reactome）</sup></sub>", ""]
    if partners:
        lines += [f"**PPI Partners ({len(partners)} genes):** " + ", ".join(f"`{p}`" for p in partners), ""]
    return "\n".join(lines)

def build_report(gene, disease, lang, hypothesis, context, generated_iso, ppi_section="", enrichment_section=""):
    parts=[f"# Drug Discovery Hypothesis: {gene} × {disease}",
           f"Generated: {generated_iso}  |  Language: {lang}", "", "---", "",
           hypothesis, ""]
    if ppi_section or enrichment_section:
        parts += ["---", "", "## Supporting Evidence", ""]
        if ppi_section: parts.append(ppi_section)
        if enrichment_section: parts.append(enrichment_section)
    parts += ["---", "", "## Evidence Context", "", context]
    return "\n".join(parts)

def summary_html(results, disease):
    rows=""
    for r in results:
        done=r.get("status","").startswith("✓")
        color="#1a7f37" if done else "#cf222e"
        rows+=(f'<tr><td style="padding:6px 14px;border:1px solid #ddd;font-weight:bold">{r["gene"]}</td>'
               f'<td style="padding:6px 14px;border:1px solid #ddd;color:{color}">{r.get("status","")}</td>'
               f'<td style="padding:6px 14px;border:1px solid #ddd;color:#555;font-size:12px">{r.get("path","")}</td></tr>')
    n=sum(1 for r in results if r.get("status","").startswith("✓"))
    return (f'<h3>Batch Summary — {disease}</h3><p>完了: {n}/{len(results)} 遺伝子</p>'
            f'<table style="border-collapse:collapse;font-size:13px"><thead><tr>'
            f'<th style="padding:6px 14px;border:1px solid #ddd;background:#f5f5f5">遺伝子</th>'
            f'<th style="padding:6px 14px;border:1px solid #ddd;background:#f5f5f5">ステータス</th>'
            f'<th style="padding:6px 14px;border:1px solid #ddd;background:#f5f5f5">レポート</th>'
            f'</tr></thead><tbody>{rows}</tbody></table>')

# ── パイプライン ────────────────────────────────────────────────────────────
def process_gene(gene, disease, disease_id, lang=LANG, context_config=None, verbose=True):
    def log(msg=""):
        if verbose: print(msg)
    # 1. データ収集
    try:
        evidence=collect_all(gene, disease, verbose=verbose, disease_id=disease_id)
    except Exception as e:
        return {"gene":gene,"status":f"データ収集失敗: {e}"}
    # 2-3. PPI + エンリッチメント
    ppi_graph=None; enrichment={}
    try:
        log("  PPIネットワーク構築中...")
        ppi_graph=build_ppi_network(gene)
        enrichment=run_network_enrichment(ppi_graph) if ppi_graph else {}
    except Exception as e:
        log(f"  ⚠ ネットワークエラー: {e}")
    # 4. コンテキスト
    context=build_llm_context(evidence, config=context_config)
    if ppi_graph:
        context += "\n\n" + network_summary_for_llm(ppi_graph, gene, enrichment)
    log(f"  コンテキスト: {len(context):,} 文字")
    # 5. 仮説生成
    log("  仮説生成中...\n")
    try:
        cb=(lambda tok: print(tok, end="", flush=True)) if verbose else None
        hypothesis=generate_hypothesis(gene, disease, context, llm, lang=lang, stream_callback=cb)
        log(f"\n  ✓ 仮説生成完了 ({len(hypothesis):,} 文字)")
    except Exception as e:
        return {"gene":gene,"status":f"仮説生成失敗: {e}"}
    # 6. 保存
    ts=datetime.now().strftime("%Y%m%d_%H%M%S")
    pair_dir=REPORTS_DIR/f"{gene}_{disease.replace(' ','_')}"
    pair_dir.mkdir(parents=True, exist_ok=True)
    ppi_section=""
    if ppi_graph and ppi_graph.number_of_edges()>0:
        img_name=f"{ts}_ppi.png"
        partners_list=rank_partners(ppi_graph, gene.upper(), max_n=30) if ppi_graph else []
        if render_ppi_image(ppi_graph, gene, str(pair_dir/img_name), enrichment=enrichment, max_nodes=30):
            ppi_section=ppi_md(gene, img_name, partners=partners_list)
    md=build_report(gene, disease, lang, hypothesis, context, datetime.now().isoformat(),
                    ppi_section=ppi_section, enrichment_section=enrichment_md(enrichment))
    rpt_path=pair_dir/f"{ts}_{'JA' if lang=='ja' else 'EN'}.md"
    rpt_path.write_text(md, encoding="utf-8")
    (pair_dir/f"{ts}_raw.json").write_text(json.dumps(evidence,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    log(f"  ✓ 保存: {rpt_path}")
    return {"gene":gene,"status":"✓ 完了","path":str(rpt_path)}


def run_batch(genes, selected_disease, lang=LANG, context_config=None, verbose=True):
    disease=selected_disease["name"]; disease_id=selected_disease["id"]
    if verbose:
        print(f"疾患: {disease}  ({disease_id})")
        print(f"対象遺伝子 ({len(genes)}件): {', '.join(genes)}")
        print("="*60)
    results=[]
    for i,gene in enumerate(genes, 1):
        if verbose: print(f"\n[{i}/{len(genes)}] {gene} × {disease}\n" + "-"*50)
        results.append(process_gene(gene, disease, disease_id, lang, context_config, verbose))
    return results

print("✓ レポート + パイプライン定義完了")

✓ レポート + パイプライン定義完了


## Step 2: 疾患名検索

疾患名を入力して検索し、リストから選択してください。

In [11]:
# ============================================================
# Step 2: 疾患名検索（OpenTargets）
# ============================================================
_OT_SEARCH_QUERY = '''
query($q:String!){
  search(queryString:$q, entityNames:["disease"], page:{index:0,size:15}) {
    hits { id name entity description }
  }
}
'''

disease_input = widgets.Text(placeholder="例: Parkinson disease", description="疾患名:", layout=widgets.Layout(width="350px"))
search_btn = widgets.Button(description="検索", button_style="primary")
disease_list_w = widgets.Select(options=[], description="疾患選択:", layout=widgets.Layout(width="600px", height="200px"))
disease_label_w = widgets.Label("")
selected_disease = {}

def on_search(b):
    q=disease_input.value.strip()
    if not q: return
    disease_label_w.value = "検索中..."
    try:
        r=requests.post(_OT_API, json={"query":_OT_SEARCH_QUERY,"variables":{"q":q}},timeout=20)
        hits=[h for h in r.json().get("data",{}).get("search",{}).get("hits",[]) if h.get("entity")=="disease"]
        disease_list_w.options=[(f"{h['name']}  [{h['id']}]", h) for h in hits]
        disease_label_w.value = f"{len(hits)} 件見つかりました"
    except Exception as e:
        disease_label_w.value = f"エラー: {e}"

def on_select(change):
    val=change["new"]
    if val:
        selected_disease.clear(); selected_disease.update({"name":val["name"],"id":val["id"]})
        disease_label_w.value = f"✓ 選択: {val['name']}  (ID: {val['id']})"

search_btn.on_click(on_search)
disease_list_w.observe(on_select, names="value")
display(widgets.VBox([widgets.HBox([disease_input, search_btn]), disease_list_w, disease_label_w]))

## Step 3: 解析遺伝子リスト入力

In [ ]:
# ============================================================
# Step 3: 遺伝子リスト入力 + HGNC 検証
# ============================================================
gene_input_w = widgets.Textarea(placeholder="例:\nBRCA1\nTP53\nEGFR", description="遺伝子:", layout=widgets.Layout(width="350px", height="120px"))
validate_btn = widgets.Button(description="HGNC検証", button_style="info")
gene_out_w = widgets.Output()
GENE_LIST = []

def validate_genes(b):
    global GENE_LIST
    raw=[g.strip().upper() for g in gene_input_w.value.replace(",","\n").splitlines() if g.strip()]
    with gene_out_w:
        gene_out_w.clear_output()
        validated=[]; invalid=[]
        for gene in raw:
            try:
                r=requests.get("https://rest.genenames.org/fetch/symbol/"+gene,headers={"Accept":"application/json"},timeout=10)
                docs=r.json().get("response",{}).get("docs",[])
                if docs: validated.append(docs[0].get("symbol",gene)); print(f"✓ {gene}")
                else: invalid.append(gene); print(f"✗ {gene} — HGNCで見つかりません")
            except: validated.append(gene); print(f"? {gene} — 検証スキップ（ネットワークエラー）")
        GENE_LIST=validated
        print(f"\n対象遺伝子 ({len(GENE_LIST)}件): {', '.join(GENE_LIST)}")

validate_btn.on_click(validate_genes)
display(widgets.VBox([widgets.HBox([gene_input_w, validate_btn]), gene_out_w]))

## Step 4: バッチ実行

In [14]:
# ============================================================
# Step 4: バッチ実行
# ============================================================
if not GENE_LIST:
    print("✗ 遺伝子リストが空です。Step 3 を実行してください。")
elif not selected_disease:
    print("✗ 疾患が選択されていません。Step 2 を実行してください。")
else:
    DISEASE = selected_disease["name"]
    print(f"開始: {len(GENE_LIST)} 遺伝子 × {DISEASE}")
    print(f"言語: {LANG}  モデル: {MODEL}\n")
    results = run_batch(GENE_LIST, selected_disease, lang=LANG, context_config=CONTEXT_CONFIG)
    display(HTML(summary_html(results, DISEASE)))
    # サマリー保存
    ts=datetime.now().strftime("%Y%m%d_%H%M%S")
    REPORTS_DIR.mkdir(exist_ok=True)
    sm_path=REPORTS_DIR/f"{DISEASE.replace(' ','_')}_summary_{ts}.md"
    n_done=sum(1 for r in results if r.get("status","").startswith("✓"))
    lines=[f"# Batch Summary — {DISEASE}",f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}","",
           f"完了: {n_done}/{len(results)} 遺伝子","","| 遺伝子 | ステータス | レポート |","|---|---|---|"]
    for r in results: lines.append(f"| {r['gene']} | {r.get('status','')} | {r.get('path','')} |")
    sm_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"\nサマリー保存: {sm_path}")

開始: 1 遺伝子 × Hepatic fibrosis
言語: en  モデル: qwen2.5:14b

疾患: Hepatic fibrosis  (HP_0001395)
対象遺伝子 (1件): ADRA1A

[1/1] ADRA1A × Hepatic fibrosis
--------------------------------------------------
    [IntAct] キャッシュ使用
  [+] intact: OK
  [+] gnomad: OK
  [+] gtex: OK
  [+] dgidb: OK
  [+] reactome: OK
  [+] clinicaltrials: OK
  [+] opentargets: OK
  [+] uniprot: OK
  [+] hpa: OK
  [+] gwas: OK
  [+] alphafold: OK
  [+] chembl: OK
  [+] clinvar: OK
    遺伝子シノニム (6): ADRA1A, Alpha-1D adrenergic receptor, Alpha-1A adrenergic receptor, Alpha-1D adrenoreceptor, Alpha-1D adrenoceptor
    疾患シノニム  (7): Hepatic fibrosis, Liver Cirrhosis, Cirrhosis, Liver, Hepatic Cirrhosis, Cirrhosis, Hepatic
  [+] pubmed: OK
  [+] toxicity: OK

  完了: 14/14 ソース
  PPIネットワーク構築中...
  IntAct 取得中...
    [IntAct] キャッシュ使用
  SIGNOR 取得中...
  Reactome 取得中...
    [Reactome] キャッシュ使用
  ネットワーク: 21 nodes / 20 edges
  エンリッチメント対象: 21 遺伝子
  エンリッチメント: 30 有意項目
  コンテキスト: 5,677 文字
  仮説生成中...

## 1. Target Validity

### 1a. Genetic Evidenc

TypeError: ppi_md() got an unexpected keyword argument 'partners'